# Encrypted Neural Network Inference Benchmark
## Fully-Connected Classification: Plaintext vs. Concrete-ML vs. TenSEAL (CKKS)

This notebook benchmarks **encrypted neural network inference** on a fully-connected (dense)
classification network using two mature FHE libraries:

| Library | Scheme | Hardware |
|---|---|---|
| **Concrete-ML** (Zama) | TFHE | CPU (GPU backend planned) |
| **TenSEAL** | CKKS | CPU-only |
| **`fhe_sdk`** *(coming)* | CKKS | GPU via HEonGPU/FIDESlib |

The goal is to:
1. Establish a plaintext accuracy baseline.
2. Measure the **accuracy loss** introduced by FHE approximations.
3. Measure **wall-clock times** for key generation, encryption, inference, and decryption.
4. Show quantitatively why GPU acceleration (the `fhe_sdk` goal) is necessary.

### Network Architecture

```
Input(64) → Linear(64→64) → Square activation (x²) → Linear(64→10) → argmax
```

Square activation is chosen because it is the most FHE-natural non-linearity —
a single multiplication that consumes exactly **one CKKS level**.

### Dataset

`sklearn.datasets.make_classification` with 1 000 samples, 64 features, 10 classes.
No download required; reproducible via `random_state=42`.

---
## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if running in a fresh environment)
# !pip install concrete-ml tenseal scikit-learn torch numpy pandas matplotlib

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import tenseal as ts

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

print("NumPy:",  np.__version__)
print("PyTorch:", torch.__version__)
print("TenSEAL:", ts.__version__)

---
## 2. Dataset

We generate a synthetic dataset that exercises the full 64-dimensional input space
expected by the network. Using `make_classification` avoids any network download and
keeps the notebook self-contained.

| Parameter | Value | Rationale |
|---|---|---|
| `n_samples` | 1 000 | Small enough for fast FHE inference on CPU |
| `n_features` | 64 | Matches the network input dimension |
| `n_classes` | 10 | 10-way classification |
| `n_informative` | 30 | Non-trivial but solvable problem |
| `random_state` | 42 | Fully reproducible |

In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=64,
    n_classes=10,
    n_informative=30,
    n_redundant=10,
    n_repeated=0,
    random_state=42,
)

# 80 / 20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise features (important for CKKS numerical stability)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")
print(f"Classes          : {len(np.unique(y_train))}")
print(f"Feature range    : [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# Small batch used for FHE inference (TenSEAL is slow on CPU)
FHE_BATCH_SIZE = 10
X_fhe = X_test[:FHE_BATCH_SIZE]
y_fhe = y_test[:FHE_BATCH_SIZE]
print(f"FHE evaluation batch: {FHE_BATCH_SIZE} samples")

---
## 3. Plaintext Baseline

We train **two** plaintext baselines so that the FHE results can be compared fairly:

1. **sklearn `MLPClassifier`** — quick reference for achievable accuracy.
2. **PyTorch MLP with Square activation** — the *exact* architecture used for TenSEAL
   encrypted inference. Using the same weights in both plaintext and FHE settings
   isolates the accuracy loss attributable purely to encryption noise and quantisation.

### 3a. sklearn MLPClassifier (reference)

In [ ]:
t0 = time.perf_counter()
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    max_iter=200,
    random_state=42,
    verbose=False,
)
sklearn_mlp.fit(X_train, y_train)
sklearn_train_time = time.perf_counter() - t0

sklearn_preds = sklearn_mlp.predict(X_test)
sklearn_acc   = accuracy_score(y_test, sklearn_preds)

print(f"sklearn MLP accuracy : {sklearn_acc:.4f}  ({sklearn_acc*100:.1f}%)")
print(f"Training time        : {sklearn_train_time:.2f}s")

### 3b. PyTorch MLP with Square Activation (FHE-compatible architecture)

The network mirrors what we will run under CKKS:
```
Input(64) → Linear(64→64) → x² → Linear(64→10)
```
Square activation is polynomial and requires **no approximation** in CKKS — it is a single
homomorphic multiplication. This keeps the CKKS depth budget at 1 level, compatible
with `poly_modulus_degree=8192`.

In [ ]:
class SquareNet(nn.Module):
    """Two-layer FC network with square activation — FHE-compatible architecture."""

    def __init__(self, in_features: int = 64, hidden: int = 64, n_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = x * x          # square activation — polynomial, CKKS depth cost = 1
        x = self.fc2(x)
        return x


def train_pytorch_model(
    model: nn.Module,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    epochs: int = 30,
    lr: float = 1e-3,
    batch_size: int = 64,
) -> list:
    """Train model with cross-entropy loss and return per-epoch loss history."""
    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    optimiser = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = []

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimiser.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item() * len(xb)
        history.append(epoch_loss / len(X_tr))

    return history


def evaluate_pytorch(model: nn.Module, X_ev: np.ndarray, y_ev: np.ndarray) -> float:
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_ev, dtype=torch.float32))
        preds  = logits.argmax(dim=1).numpy()
    return accuracy_score(y_ev, preds)

In [ ]:
pytorch_model = SquareNet(in_features=64, hidden=64, n_classes=10)

t0 = time.perf_counter()
loss_history = train_pytorch_model(pytorch_model, X_train, y_train, epochs=30)
pytorch_train_time = time.perf_counter() - t0

pytorch_acc = evaluate_pytorch(pytorch_model, X_test, y_test)

print(f"PyTorch SquareNet accuracy : {pytorch_acc:.4f}  ({pytorch_acc*100:.1f}%)")
print(f"Training time              : {pytorch_train_time:.2f}s")
print(f"Final training loss        : {loss_history[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(loss_history, linewidth=1.8, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('PyTorch SquareNet — Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plaintext inference time on the FHE batch (for a fair timing comparison later)
pytorch_model.eval()
X_fhe_tensor = torch.tensor(X_fhe, dtype=torch.float32)

# Warm-up
with torch.no_grad():
    _ = pytorch_model(X_fhe_tensor)

N_REPEATS = 50
t0 = time.perf_counter()
for _ in range(N_REPEATS):
    with torch.no_grad():
        pt_logits = pytorch_model(X_fhe_tensor)
plaintext_inference_time = (time.perf_counter() - t0) / N_REPEATS

pt_preds_fhe_batch = pt_logits.argmax(dim=1).numpy()
plaintext_batch_acc = accuracy_score(y_fhe, pt_preds_fhe_batch)

print(f"Plaintext inference ({FHE_BATCH_SIZE} samples): {plaintext_inference_time*1000:.4f} ms")
print(f"Plaintext accuracy on FHE batch     : {plaintext_batch_acc:.4f}")

---
## 4. Concrete-ML Implementation

[Concrete-ML](https://github.com/zama-ai/concrete-ml) by Zama compiles sklearn/PyTorch
models to circuits that execute under **TFHE** — a Boolean-gate-level FHE scheme.
Unlike CKKS, TFHE works on integers and requires **quantisation** of all weights and
activations to `n_bits` precision.

We use `NeuralNetClassifier` which wraps a Brevitas-quantised PyTorch network and
drives the Concrete compiler. The same `SquareNet` topology is replicated via
a Brevitas `QuantLinear` + polynomial activation.

> **Note:** Concrete-ML uses TFHE (not CKKS). It is included here because it is the
> dominant alternative for FHE-ML on the GPU. `fhe_sdk` will target CKKS on GPU,
> filling the gap that Concrete-ML leaves.

In [ ]:
# ── Try to import Concrete-ML. Gracefully skip if not installed. ──────────────
try:
    from concrete.ml.sklearn import NeuralNetClassifier
    from torch import nn as torch_nn
    import brevitas.nn as qnn
    CONCRETE_AVAILABLE = True
    print("Concrete-ML import: OK")
except ImportError as exc:
    CONCRETE_AVAILABLE = False
    print(f"Concrete-ML not available ({exc}).")
    print("Skipping Concrete-ML section. Install with: pip install concrete-ml")

In [ ]:
concrete_results = {
    'compile_time_s':  None,
    'inference_time_s': None,
    'accuracy':        None,
    'n_bits':          None,
}

if CONCRETE_AVAILABLE:
    N_BITS = 6  # quantisation bit-width — tradeoff: accuracy ↑ vs speed ↓

    # Concrete-ML NeuralNetClassifier accepts a module factory callable
    # that returns a Brevitas-compatible nn.Module.
    # We build a simple two-layer network with x^2 = QuantIdentity output * same.
    # For Concrete-ML we use a degree-2 polynomial approximation of the identity
    # so the compiler can lower it to integer arithmetic.

    class ConcreteSquareNet(torch_nn.Module):
        """FHE-friendly FC network for Concrete-ML compilation."""

        def __init__(self):
            super().__init__()
            self.fc1 = qnn.QuantLinear(64, 64, bias=True,
                                       weight_bit_width=N_BITS,
                                       bias_quant=None)
            self.act  = torch_nn.ReLU()  # Concrete-ML supports ReLU natively
            self.fc2  = qnn.QuantLinear(64, 10, bias=True,
                                        weight_bit_width=N_BITS,
                                        bias_quant=None)

        def forward(self, x):
            return self.fc2(self.act(self.fc1(x)))

    concrete_clf = NeuralNetClassifier(
        module=ConcreteSquareNet,
        max_epochs=20,
        lr=5e-3,
        iterator_train__shuffle=True,
        verbose=0,
        n_bits=N_BITS,
    )

    print("Training Concrete-ML NeuralNetClassifier...")
    t0 = time.perf_counter()
    concrete_clf.fit(X_train, y_train)
    concrete_train_time = time.perf_counter() - t0
    print(f"  Training time: {concrete_train_time:.2f}s")

    # ── Compile to FHE ──────────────────────────────────────────────────────
    print("Compiling to FHE circuit...")
    t0 = time.perf_counter()
    concrete_clf.compile(X_train)
    compile_time = time.perf_counter() - t0
    print(f"  Compile time: {compile_time:.2f}s")

    # ── Encrypted inference on FHE batch ──────────────────────────────────
    print(f"Running encrypted inference on {FHE_BATCH_SIZE} samples...")
    t0 = time.perf_counter()
    concrete_preds = concrete_clf.predict(X_fhe, fhe='execute')
    concrete_infer_time = time.perf_counter() - t0
    print(f"  Inference time ({FHE_BATCH_SIZE} samples): {concrete_infer_time:.2f}s")
    print(f"  Per-sample time: {concrete_infer_time/FHE_BATCH_SIZE:.2f}s")

    concrete_acc = accuracy_score(y_fhe, concrete_preds)
    print(f"  FHE Accuracy: {concrete_acc:.4f}")

    concrete_results.update({
        'compile_time_s':   compile_time,
        'inference_time_s': concrete_infer_time / FHE_BATCH_SIZE,
        'accuracy':         concrete_acc,
        'n_bits':           N_BITS,
    })
else:
    print("Concrete-ML section skipped.")

---
## 5. TenSEAL Implementation (CKKS)

[TenSEAL](https://github.com/OpenMined/TenSEAL) implements CKKS on the CPU using
Microsoft SEAL as its backend. We manually run the pre-trained `SquareNet` weights
through TenSEAL's `CKKSVector` API.

### CKKS Parameter Choice

| Parameter | Value | Rationale |
|---|---|---|
| `poly_modulus_degree` | 8 192 | Supports vectors up to 4 096 slots, 128-bit security |
| `coeff_mod_bit_sizes` | [60, 40, 40, 60] | 2 usable levels: 1 for Square + 1 safety margin |
| `scale` | 2^40 | Matches interior 40-bit primes to minimise rescaling noise |

The depth budget is:
```
usable_levels = len(coeff_mod_bit_sizes) - 2 = 2
Square activation consumes 1 level → 1 level of margin remains.
```

### Inference Flow

```
x (plaintext list[float])
  → ts.ckks_vector(ctx, x)        # encrypt
  → enc_x.linear(W1, b1)          # Linear(64→64): plaintext weights, no level consumed
  → enc_x * enc_x                 # Square: consumes 1 CKKS level
  → enc_x.linear(W2, b2)          # Linear(64→10): no level consumed
  → enc_x.decrypt()               # decrypt
  → argmax(result[:10])           # predicted class
```

### 5a. CKKS Context & Key Generation

In [ ]:
POLY_MOD_DEGREE    = 8192
COEFF_MOD_BIT_SIZES = [60, 40, 40, 60]
SCALE              = 2**40

print("Creating TenSEAL CKKS context and generating keys...")
t0 = time.perf_counter()

ts_context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=POLY_MOD_DEGREE,
    coeff_mod_bit_sizes=COEFF_MOD_BIT_SIZES,
)
ts_context.generate_galois_keys()    # needed for rotation / linear transforms
ts_context.global_scale = SCALE

keygen_time = time.perf_counter() - t0

print(f"Key generation time : {keygen_time:.4f}s")
print(f"Max slots           : {POLY_MOD_DEGREE // 2}")
print(f"Usable CKKS levels  : {len(COEFF_MOD_BIT_SIZES) - 2}")

### 5b. Extract Weights from Trained PyTorch Model

In [ ]:
def extract_layer_params(linear_layer: nn.Linear):
    """Return (weight, bias) as plain Python lists (row-major)."""
    W = linear_layer.weight.detach().numpy().tolist()   # shape [out, in]
    b = linear_layer.bias.detach().numpy().tolist()     # shape [out]
    return W, b


W1, b1 = extract_layer_params(pytorch_model.fc1)
W2, b2 = extract_layer_params(pytorch_model.fc2)

print(f"Layer 1 weights: {len(W1)} x {len(W1[0])}")
print(f"Layer 2 weights: {len(W2)} x {len(W2[0])}")

### 5c. TenSEAL Encrypted Inference

In [ ]:
def tenseal_forward(x_plain: list, context: ts.Context,
                    W1: list, b1: list,
                    W2: list, b2: list) -> tuple:
    """
    Run a single sample through the 2-layer SquareNet under CKKS encryption.

    Returns
    -------
    predicted_class : int
    timing_breakdown : dict with keys encrypt_s, infer_s, decrypt_s
    """
    # --- Encryption -------------------------------------------------------
    t_enc_start = time.perf_counter()
    enc_x = ts.ckks_vector(context, x_plain)
    t_enc_end = time.perf_counter()

    # --- Homomorphic inference -------------------------------------------
    t_infer_start = time.perf_counter()

    # Linear layer 1: enc_x @ W1^T + b1
    # TenSEAL matrix_vector_product applies W row-wise
    enc_x = enc_x.linear(W1, b1)

    # Square activation: x^2 — consumes 1 CKKS level
    enc_x = enc_x * enc_x

    # Linear layer 2: enc_x @ W2^T + b2
    enc_x = enc_x.linear(W2, b2)

    t_infer_end = time.perf_counter()

    # --- Decryption -------------------------------------------------------
    t_dec_start = time.perf_counter()
    raw_output = enc_x.decrypt()
    t_dec_end   = time.perf_counter()

    # Only the first 10 slots are valid output logits
    logits = raw_output[:10]
    predicted_class = int(np.argmax(logits))

    timing = {
        'encrypt_s': t_enc_end   - t_enc_start,
        'infer_s':   t_infer_end - t_infer_start,
        'decrypt_s': t_dec_end   - t_dec_start,
    }
    return predicted_class, timing

In [ ]:
print(f"Running TenSEAL CKKS inference on {FHE_BATCH_SIZE} samples...")
print("(This is CPU-only CKKS — expect several seconds per sample)\n")

ts_preds   = []
ts_timings = []

for i, x_sample in enumerate(X_fhe):
    pred, timing = tenseal_forward(
        x_sample.tolist(), ts_context, W1, b1, W2, b2
    )
    ts_preds.append(pred)
    ts_timings.append(timing)

    total_sample = timing['encrypt_s'] + timing['infer_s'] + timing['decrypt_s']
    print(
        f"  Sample {i+1:2d}/{FHE_BATCH_SIZE} | "
        f"true={y_fhe[i]} pred={pred} | "
        f"enc={timing['encrypt_s']*1000:.1f}ms  "
        f"infer={timing['infer_s']*1000:.1f}ms  "
        f"dec={timing['decrypt_s']*1000:.1f}ms  "
        f"total={total_sample*1000:.1f}ms"
    )

ts_acc = accuracy_score(y_fhe, ts_preds)
print(f"\nTenSEAL CKKS accuracy on FHE batch: {ts_acc:.4f}")

In [ ]:
# Aggregate timing statistics
enc_times   = [t['encrypt_s']  for t in ts_timings]
infer_times = [t['infer_s']    for t in ts_timings]
dec_times   = [t['decrypt_s']  for t in ts_timings]
total_times = [e + i + d for e, i, d in zip(enc_times, infer_times, dec_times)]

ts_avg_encrypt = np.mean(enc_times)
ts_avg_infer   = np.mean(infer_times)
ts_avg_decrypt = np.mean(dec_times)
ts_avg_total   = np.mean(total_times)

print("=== TenSEAL Per-Sample Timing (averaged over batch) ===")
print(f"  Encrypt  : {ts_avg_encrypt*1000:8.2f} ms")
print(f"  Inference: {ts_avg_infer*1000:8.2f} ms")
print(f"  Decrypt  : {ts_avg_decrypt*1000:8.2f} ms")
print(f"  Total    : {ts_avg_total*1000:8.2f} ms")
print(f"\n  Compared to plaintext ({plaintext_inference_time*1000:.4f} ms per batch):")
print(f"  Slowdown factor: {ts_avg_total / (plaintext_inference_time / FHE_BATCH_SIZE):.1f}x")

### 5d. Verify Numerical Correctness

We compare the decrypted output logits against plaintext logits to confirm that
CKKS noise stays within acceptable bounds.

In [ ]:
# Rerun one sample to collect both plaintext and encrypted logits
sample_idx = 0
x_sample   = X_fhe[sample_idx].tolist()

# Plaintext logits
with torch.no_grad():
    pt_out = pytorch_model(torch.tensor(x_sample, dtype=torch.float32)).numpy()

# Encrypted logits (decrypt gives the full slot vector; first 10 are logits)
enc_vec = ts.ckks_vector(ts_context, x_sample)
enc_vec = enc_vec.linear(W1, b1)
enc_vec = enc_vec * enc_vec
enc_vec = enc_vec.linear(W2, b2)
raw     = enc_vec.decrypt()
fhe_out = np.array(raw[:10])

err = np.abs(pt_out - fhe_out)
print("Plaintext logits  :", np.round(pt_out, 4))
print("CKKS logits       :", np.round(fhe_out, 4))
print(f"Max absolute error: {err.max():.6f}")
print(f"Mean absolute error: {err.mean():.6f}")
print(f"Plaintext argmax  : {np.argmax(pt_out)}")
print(f"CKKS argmax       : {np.argmax(fhe_out)}")

---
## 6. Benchmark Comparison

This section aggregates all measurements into a unified table and set of charts
to support the thesis comparison between TenSEAL (CPU CKKS), Concrete-ML (CPU/GPU TFHE),
and the forthcoming `fhe_sdk` (GPU CKKS).

In [ ]:
# ── Build results table ───────────────────────────────────────────────────────

rows = []

# Plaintext baseline
rows.append({
    'System':             'Plaintext (PyTorch)',
    'Scheme':             'None',
    'Hardware':           'CPU',
    'Activation':         'Square (x²)',
    'Accuracy (batch)':   f'{plaintext_batch_acc:.4f}',
    'Key Gen (s)':        'N/A',
    'Encrypt/sample (ms)':'N/A',
    'Infer/sample (ms)':  f'{(plaintext_inference_time/FHE_BATCH_SIZE)*1000:.4f}',
    'Decrypt/sample (ms)':'N/A',
    'Total/sample (ms)':  f'{(plaintext_inference_time/FHE_BATCH_SIZE)*1000:.4f}',
    'Accuracy Loss':      '—',
})

# TenSEAL
rows.append({
    'System':             'TenSEAL',
    'Scheme':             'CKKS',
    'Hardware':           'CPU',
    'Activation':         'Square (x²)',
    'Accuracy (batch)':   f'{ts_acc:.4f}',
    'Key Gen (s)':        f'{keygen_time:.4f}',
    'Encrypt/sample (ms)':f'{ts_avg_encrypt*1000:.2f}',
    'Infer/sample (ms)':  f'{ts_avg_infer*1000:.2f}',
    'Decrypt/sample (ms)':f'{ts_avg_decrypt*1000:.2f}',
    'Total/sample (ms)':  f'{ts_avg_total*1000:.2f}',
    'Accuracy Loss':      f'{abs(plaintext_batch_acc - ts_acc):.4f}',
})

# Concrete-ML (if available)
if CONCRETE_AVAILABLE and concrete_results['accuracy'] is not None:
    rows.append({
        'System':             f'Concrete-ML (n_bits={concrete_results["n_bits"]})',
        'Scheme':             'TFHE',
        'Hardware':           'CPU',
        'Activation':         'ReLU (approx)',
        'Accuracy (batch)':   f'{concrete_results["accuracy"]:.4f}',
        'Key Gen (s)':        f'{concrete_results["compile_time_s"]:.2f} (compile)',
        'Encrypt/sample (ms)':'—',
        'Infer/sample (ms)':  f'{concrete_results["inference_time_s"]*1000:.2f}',
        'Decrypt/sample (ms)':'—',
        'Total/sample (ms)':  f'{concrete_results["inference_time_s"]*1000:.2f}',
        'Accuracy Loss':      f'{abs(plaintext_batch_acc - concrete_results["accuracy"]):.4f}',
    })
else:
    rows.append({
        'System':             'Concrete-ML',
        'Scheme':             'TFHE',
        'Hardware':           'CPU',
        'Activation':         'ReLU (approx)',
        'Accuracy (batch)':   'N/A (not installed)',
        'Key Gen (s)':        'N/A',
        'Encrypt/sample (ms)':'N/A',
        'Infer/sample (ms)':  'N/A',
        'Decrypt/sample (ms)':'N/A',
        'Total/sample (ms)':  'N/A',
        'Accuracy Loss':      'N/A',
    })

# fhe_sdk placeholder (to be filled in once GPU backend is available)
rows.append({
    'System':             'fhe_sdk (planned)',
    'Scheme':             'CKKS',
    'Hardware':           'GPU (CUDA)',
    'Activation':         'Square (x²)',
    'Accuracy (batch)':   'TBD',
    'Key Gen (s)':        'TBD',
    'Encrypt/sample (ms)':'TBD',
    'Infer/sample (ms)':  'TBD',
    'Decrypt/sample (ms)':'TBD',
    'Total/sample (ms)':  'TBD',
    'Accuracy Loss':      'TBD',
})

df = pd.DataFrame(rows)
df = df.set_index('System')

print("=== Benchmark Summary ===")
pd.set_option('display.max_colwidth', None)
df

In [ ]:
# ── Timing breakdown chart ────────────────────────────────────────────────────
# We visualise the TenSEAL per-operation breakdown and the plaintext reference.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Left: per-operation stacked bar (TenSEAL) ---
ax = axes[0]
categories = ['Encrypt', 'Inference', 'Decrypt']
values_ms  = [
    ts_avg_encrypt * 1000,
    ts_avg_infer   * 1000,
    ts_avg_decrypt * 1000,
]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(categories, values_ms, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, values_ms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f} ms', ha='center', va='bottom', fontsize=9)
ax.set_title('TenSEAL CKKS — Per-Sample Breakdown')
ax.set_ylabel('Time (ms)')
ax.grid(axis='y', alpha=0.3)

# --- Right: total inference time comparison (log scale) ---
ax2 = axes[1]
systems  = ['Plaintext\n(PyTorch)', 'TenSEAL\n(CKKS CPU)']
totals_s = [
    plaintext_inference_time / FHE_BATCH_SIZE * 1000,
    ts_avg_total * 1000,
]
bar_colors = ['#4C72B0', '#DD8452']
if CONCRETE_AVAILABLE and concrete_results['inference_time_s'] is not None:
    systems.append('Concrete-ML\n(TFHE CPU)')
    totals_s.append(concrete_results['inference_time_s'] * 1000)
    bar_colors.append('#55A868')

systems.append('fhe_sdk\n(CKKS GPU)')
totals_s.append(None)  # placeholder
bar_colors.append('#C44E52')

# Filter out None placeholders for plotting
plot_systems = [s for s, t in zip(systems, totals_s) if t is not None]
plot_totals  = [t for t in totals_s if t is not None]
plot_colors  = [c for c, t in zip(bar_colors, totals_s) if t is not None]

bars2 = ax2.bar(plot_systems, plot_totals, color=plot_colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, plot_totals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
             f'{val:.1f} ms', ha='center', va='bottom', fontsize=9)
ax2.set_yscale('log')
ax2.set_title('Total Inference Time per Sample (log scale)')
ax2.set_ylabel('Time (ms, log scale)')
ax2.grid(axis='y', alpha=0.3, which='both')
ax2.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax2.annotate('GPU target\n(fhe_sdk)',
             xy=(len(plot_systems) - 0.3, min(plot_totals)),
             fontsize=8, color='#C44E52', ha='right')

plt.tight_layout()
plt.savefig('benchmark_timing.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to benchmark_timing.png")

In [ ]:
# ── Accuracy comparison chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

acc_systems = ['Plaintext\n(PyTorch)', 'TenSEAL\n(CKKS CPU)']
acc_values  = [plaintext_batch_acc, ts_acc]
acc_colors  = ['#4C72B0', '#DD8452']

if CONCRETE_AVAILABLE and concrete_results['accuracy'] is not None:
    acc_systems.append('Concrete-ML\n(TFHE CPU)')
    acc_values.append(concrete_results['accuracy'])
    acc_colors.append('#55A868')

bars_acc = ax.bar(acc_systems, acc_values, color=acc_colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars_acc, acc_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 1.15)
ax.axhline(plaintext_batch_acc, linestyle='--', color='grey', linewidth=1, label='Plaintext reference')
ax.set_title(f'Accuracy on FHE Batch ({FHE_BATCH_SIZE} samples)')
ax.set_ylabel('Accuracy')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to benchmark_accuracy.png")

In [ ]:
# ── Print full numeric summary ─────────────────────────────────────────────────
pt_per_sample_ms = (plaintext_inference_time / FHE_BATCH_SIZE) * 1000
slowdown         = ts_avg_total / (plaintext_inference_time / FHE_BATCH_SIZE)

print("=" * 60)
print("FINAL BENCHMARK SUMMARY")
print("=" * 60)
print(f"Dataset : make_classification (1000 samples, 64 features, 10 classes)")
print(f"Network : Input(64) -> Linear(64->64) -> Square -> Linear(64->10)")
print(f"Batch   : {FHE_BATCH_SIZE} samples evaluated under FHE")
print()
print("Accuracy")
print(f"  sklearn MLP (relu, full test set)  : {sklearn_acc:.4f}")
print(f"  PyTorch SquareNet (full test set)  : {pytorch_acc:.4f}")
print(f"  PyTorch SquareNet (FHE batch)      : {plaintext_batch_acc:.4f}")
print(f"  TenSEAL CKKS      (FHE batch)      : {ts_acc:.4f}")
print()
print("TenSEAL CKKS Timing (per sample)")
print(f"  Key generation    : {keygen_time:.4f} s")
print(f"  Encryption        : {ts_avg_encrypt*1000:.2f} ms")
print(f"  Inference         : {ts_avg_infer*1000:.2f} ms")
print(f"  Decryption        : {ts_avg_decrypt*1000:.2f} ms")
print(f"  Total             : {ts_avg_total*1000:.2f} ms")
print()
print(f"Plaintext inference (per sample): {pt_per_sample_ms:.4f} ms")
print(f"TenSEAL vs plaintext slowdown   : {slowdown:.1f}x")
print("=" * 60)

---
## 7. Notes on `fhe_sdk` (GPU-Accelerated CKKS)

The benchmarks above run entirely on CPU. TenSEAL demonstrates that CKKS inference
is **correct** — the decrypted logits match the plaintext logits within negligible
floating-point noise — but the latency overhead is massive: several hundred milliseconds
per sample versus sub-millisecond plaintext inference.

### Why GPU acceleration is essential

The dominant cost in CKKS inference is the **Number Theoretic Transform (NTT)**, which
underpins every polynomial multiplication. For `poly_modulus_degree = 8192`, each
homomorphic multiplication requires O(N log N) NTT operations where N = 8192.
GPUs are extremely well-suited to NTT because it maps to a regular butterfly
network with high data parallelism and no branch divergence.

Published results from HEonGPU and FIDESlib show **10–100x speedups** over
CPU-only SEAL/TenSEAL for single-ciphertext CKKS operations on modern NVIDIA GPUs.

### How `fhe_sdk` will replace TenSEAL in this notebook

Once the GPU backend is built and the Python bindings are complete, the TenSEAL
section above will be replaced by the following drop-in equivalent:

```python
# fhe_sdk GPU-accelerated CKKS inference (replaces TenSEAL section)
from fhe_sdk import FHEContext
from fhe_sdk.nn import Sequential, Linear, Square

ctx = FHEContext.default()          # poly_modulus_degree=8192, scale=2**40

fhe_model = Sequential(
    Linear(in_features=64, out_features=64),
    Square(),
    Linear(in_features=64, out_features=10),
)
fhe_model[0].load_from_torch(pytorch_model.fc1)
fhe_model[2].load_from_torch(pytorch_model.fc2)

# Encrypted inference — dispatched to GPU via HEonGPU / FIDESlib
enc_input  = ctx.encrypt(X_fhe[0].tolist())
enc_output = fhe_model(enc_input)
result     = enc_output.decrypt()           # list[float] of length 10
predicted_class = int(np.argmax(result[:10]))
```

The API is identical to the plaintext PyTorch code up to the `ctx.encrypt()` call.
Users never interact with raw `CKKSCiphertext` objects or manage CKKS levels manually.

### Comparison axes for the thesis

| Axis | TenSEAL (CPU) | Concrete-ML (CPU/GPU) | `fhe_sdk` (GPU, target) |
|---|---|---|---|
| **Scheme** | CKKS | TFHE | CKKS |
| **Floating-point friendly** | Yes | No (integer only) | Yes |
| **Inference latency** | High (s/sample) | Very high (s/sample) | Low (ms/sample, projected) |
| **Activation support** | Polynomial only | Any (via garbled circuits) | Square, ApproxReLU, ApproxSigmoid |
| **Hardware** | CPU | CPU (+GPU for some ops) | GPU (CUDA) |
| **Python API** | Low-level vectors | sklearn-compatible | PyTorch-like |

> The TenSEAL and Concrete-ML numbers captured in this notebook serve as the
> **CPU baseline** against which `fhe_sdk` GPU performance will be reported in
> the thesis experimental chapter.